# Упражнение: LIME вручную: взвешенный локальный линейный суррогат

## Задание

«Чёрный ящик» задан формулой f(x) = σ(2·x₁² + x₂ − x₁·x₃ − 1), где σ(t) = 1 / (1 + e⁻ᵗ).
В `input.csv` — три объекта для объяснения (`id, x1, x2, x3`), в `noise.csv` — 1000 строк стандартного
нормального шума `e1, e2, e3` (общие для всех объектов, чтобы результат был воспроизводимым).
Масштабы признаков (std на обучении): s = (1.0, 0.5, 2.0).

Для каждого объекта x₀ выполните шаги LIME:

1. Возмущения: zₖ = x₀ + eₖ ⊙ s (поэлементно); стандартизованные смещения — сами eₖ.
2. Ответы чёрного ящика: pₖ = f(zₖ).
3. Веса близости: πₖ = exp(−‖eₖ‖² / σ²), ширина ядра σ = 0.75·√3 (по умолчанию библиотеки lime).
4. Взвешенная линейная регрессия p ≈ w₀ + w₁·e₁ + w₂·e₂ + w₃·e₃ **без регуляризации**, с весами π
   (взвешенный МНК; совпадает с `LinearRegression().fit(E, p, sample_weight=π)`).
5. Качество суррогата — взвешенный R²: 1 − Σπ(p − p̂)² / Σπ(p − p̄_π)², где p̄_π — взвешенное среднее p.

## Подсказки

- Коэффициент wⱼ — изменение вероятности при сдвиге признака j на одно стандартное отклонение вблизи x₀.
  Для гладкой f и узкого ядра он близок к ∂f/∂xⱼ · sⱼ в точке x₀. Сравните: при широком ядре (как здесь
  у объекта 1) расхождение заметно — суррогат усредняет наклон по всей окрестности.
- Взвешенный МНК через обычный: умножьте строки [1, E] и p на √π и решите `np.linalg.lstsq`.
- `LinearRegression(...).score(E, p, sample_weight=π)` возвращает ровно взвешенный R² из шага 5.
  Объект 3 лежит в сильно нелинейной области — посмотрите, насколько там ниже R².

## Ожидаемый формат выходного файла

CSV с колонками `id, f_x0, intercept, w1, w2, w3, r2` — 3 строки в порядке `id`;
`f_x0` = f(x₀); значения округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/lime
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.